# 📱 CockpitSentinel: Train Balanced Distraction Model (Phone & Smoking)
### High-Precision Multi-Class Driver Distraction Detection with YOLO11s

This notebook automatically downloads and merges two curated datasets:
1. **Base In-Cabin Dataset:** `satyapal-gaikwad/driver-smoking-phoneuse-earphone-detection-9nlii-oscoe` (1,253 images with 613 high-quality smoking instances)
2. **Dedicated Driver Phone Dataset:** `long-itwjh/using-phone-while-driving-qcvye` (v2: 886 in-cabin images with 2,757 phone annotations)

Then it trains **YOLO11s** on the merged dataset and exports to **Intel OpenVINO FP16** and **ONNX**.

In [ ]:
# Step 1: Verify GPU & Install dependencies
!nvidia-smi
!pip install --upgrade ultralytics roboflow openvino onnx pyyaml


## Step 2: Download Base Smoking & Phone Datasets
We use Roboflow API to download both datasets in `yolov11` format.

In [ ]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = "PB8gWnTTkNlkxmSOpsXH"
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# 1. Download Base Smoking Dataset
print("Downloading Base Smoking Dataset...")
base_proj = rf.workspace("satyapal-gaikwad").project("driver-smoking-phoneuse-earphone-detection-9nlii-oscoe")
base_ds = base_proj.version(1).download(model_format="yolov11", location="/content/base_dataset")
print("Base dataset downloaded!")

# 2. Download Dedicated Driver Phone Dataset (v2 has 2,757 phone annotations)
print("\nDownloading Dedicated Driver Phone Dataset (long-itwjh / v2)...")
phone_proj = rf.workspace("long-itwjh").project("using-phone-while-driving-qcvye")
phone_ds = phone_proj.version(2).download(model_format="yolov11", location="/content/phone_dataset")
print("Phone dataset downloaded successfully!")


## Step 3: Merge & Standardize Classes into Unified YOLO Format
Unified Target Schema:
- `0: earphone`
- `1: person`
- `2: smoke` (cigarette, vape, smoking)
- `3: using_phone` (phone, cellphone, mobile-use)

In [ ]:
import os, shutil, yaml
from pathlib import Path
from collections import Counter

STANDARD_NAMES = ['earphone', 'person', 'smoke', 'using_phone']
STANDARD_MAP = {
    'earphone': 0, 'headphone': 0, 'earphones': 0,
    'person': 1, 'driver': 1,
    'smoke': 2, 'smoking': 2, 'cigarette': 2, 'vape': 2,
    'using_phone': 3, 'phone': 3, 'cellphone': 3, 'mobile': 3, 'mobile-use': 3, 'cell phone': 3
}

output_dir = Path("/content/merged_dataset")
if output_dir.exists():
    shutil.rmtree(output_dir)

for s in ['train', 'valid']:
    (output_dir / s / 'images').mkdir(parents=True, exist_ok=True)
    (output_dir / s / 'labels').mkdir(parents=True, exist_ok=True)

def merge_folder(src_dir, prefix, split, stats):
    yaml_file = Path(f"{src_dir}/data.yaml")
    if not yaml_file.exists():
        print(f"Warning: {yaml_file} not found")
        return 0
    with open(yaml_file, 'r') as f:
        cfg = yaml.safe_load(f)
    names = cfg['names'] if isinstance(cfg['names'], list) else [cfg['names'][i] for i in sorted(cfg['names'])]
    
    img_dir = Path(f"{src_dir}/{split}/images")
    lbl_dir = Path(f"{src_dir}/{split}/labels")
    if not img_dir.exists():
        img_dir = Path(f"{src_dir}/{split}")
    if not lbl_dir.exists():
        lbl_dir = Path(f"{src_dir}/{split}")
        
    copied = 0
    for img in img_dir.glob("*.*"):
        lbl = lbl_dir / f"{img.stem}.txt"
        if not lbl.exists():
            continue
        with open(lbl, 'r', encoding='utf-8') as f:
            lines = [l.strip().split() for l in f.readlines() if l.strip()]
        remapped = []
        for p in lines:
            cid = int(p[0])
            if cid < len(names):
                cname = str(names[cid]).lower().strip()
                if cname in STANDARD_MAP:
                    target_id = STANDARD_MAP[cname]
                    remapped.append(f"{target_id} " + " ".join(p[1:]))
                    stats[target_id] += 1
        if remapped:
            dest_img = output_dir / split / 'images' / f"{prefix}_{img.name}"
            dest_lbl = output_dir / split / 'labels' / f"{prefix}_{img.stem}.txt"
            shutil.copy(img, dest_img)
            with open(dest_lbl, 'w', encoding='utf-8') as f:
                f.write("\n".join(remapped) + "\n")
            copied += 1
    return copied

train_stats = Counter()
val_stats = Counter()

for s in ['train', 'valid']:
    st = train_stats if s == 'train' else val_stats
    c_base = merge_folder("/content/base_dataset", "base", s, st)
    c_phone = merge_folder("/content/phone_dataset", "phone", s, st)
    print(f"Split '{s}': {c_base} base images + {c_phone} phone images = {c_base + c_phone} total")

merged_yaml = {
    'path': '/content/merged_dataset',
    'train': 'train/images',
    'val': 'valid/images',
    'nc': len(STANDARD_NAMES),
    'names': STANDARD_NAMES
}
with open("/content/merged_dataset/data.yaml", 'w') as f:
    yaml.dump(merged_yaml, f)

print("\n=== MERGED DATASET CLASS COUNTS (TRAIN) ===")
for idx, name in enumerate(STANDARD_NAMES):
    print(f"  Class {idx} ({name:<12s}): {train_stats[idx]:5d} boxes")
print("\n=== MERGED DATASET CLASS COUNTS (VALID) ===")
for idx, name in enumerate(STANDARD_NAMES):
    print(f"  Class {idx} ({name:<12s}): {val_stats[idx]:5d} boxes")


## Step 4: Train YOLO11s on the Merged Dataset

In [ ]:
from ultralytics import YOLO

# Initialize pretrained YOLO11 Small model
model = YOLO("yolo11s.pt")

# Fine-tune on the balanced distraction super-dataset
results = model.train(
    data="/content/merged_dataset/data.yaml",
    epochs=60,
    imgsz=640,
    batch=16,
    patience=15,
    optimizer="AdamW",
    lr0=0.001,
    device=0,
    name="yolo11s_balanced_distraction",
    project="runs"
)

print("Training Complete! Best checkpoint at:", f"{results.save_dir}/weights/best.pt")


## Step 5: Evaluate & Print Per-Class Performance

In [ ]:
best_weights = f"{results.save_dir}/weights/best.pt"
trained_model = YOLO(best_weights)
metrics = trained_model.val(data="/content/merged_dataset/data.yaml")

print("\n=== MULTI-CLASS PERFORMANCE SUMMARY ===")
print(f"Overall Precision (P): {metrics.box.mp:.4f}")
print(f"Overall Recall (R):    {metrics.box.mr:.4f}")
print(f"Overall mAP@50:        {metrics.box.map50:.4f}")
print(f"Overall mAP@50-95:     {metrics.box.map:.4f}")


## Step 6: Export to OpenVINO (Laptop CPU) & ONNX (Vehicle Edge) and Download

In [ ]:
# 1. Export Intel OpenVINO FP16 (for laptop CPU)
ov_dir = trained_model.export(format="openvino", half=True)
print("Exported OpenVINO:", ov_dir)

# 2. Export ONNX (for NVIDIA Jetson or vehicle edge accelerator)
onnx_file = trained_model.export(format="onnx", dynamic=True)
print("Exported ONNX:", onnx_file)

# 3. Package and trigger browser download
import shutil
from google.colab import files

pkg_dir = "/content/distraction_weights_package"
os.makedirs(pkg_dir, exist_ok=True)
shutil.copy(best_weights, f"{pkg_dir}/best.pt")
shutil.copytree(str(ov_dir), f"{pkg_dir}/best_openvino_model", dirs_exist_ok=True)
shutil.copy(str(onnx_file), f"{pkg_dir}/best.onnx")

archive = shutil.make_archive("/content/cockpit_sentinel_balanced_distraction", "zip", pkg_dir)
print("Download archive ready:", archive)
files.download(archive)
